# Part III-c reported probes: Llama-3.1-8B (Colab A100 80 GB)

Runs `weight_observer.probes` (commit `b4646a9e11e3`) for the 8B: a GPTQ code cache made once by the harness's own path,
**`verify` (must reproduce the registered arms bit for bit, or everything stops)**, then the 64-step local search, the
flatness curve (38 plans) and the single-matrix sweep (457). Nothing depends on earlier runs of any notebook.

Inputs: the 8B's registered results in Drive (`MyDrive/tqp-weights/codec/llama3.1-8b/a1f13010bf1d`). Outputs:
`MyDrive/tqp-weights/probes/llama3.1-8b`, saved one measurement at a time (a rerun resumes).

**Runtime: A100 80 GB (High-RAM).** Then Runtime > Run all. ~15 min setup, ~15-20 min cache, ~4 h measurements.

In [ ]:
# 1. GPU: the A100 80 GB, checked before anything is downloaded.
import os, subprocess, sys
TEST = os.environ.get("NB_TEST") == "1"
if not TEST:
    import torch
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout)
    gib = torch.cuda.get_device_properties(0).total_memory / 2**30
    print(f"{gib:.1f} GiB")
    assert gib > 70, "not the 80 GB A100: Runtime > Disconnect and delete runtime, then reconnect"

In [ ]:
# 2. Paths, and Google Drive (registered 8B results in, probe outputs out).
ROOT = os.environ.get("NB_ROOT", "/content")
REG = os.environ.get("NB_REG", "/content/drive/MyDrive/tqp-weights/codec/llama3.1-8b/a1f13010bf1d")
OUT = os.environ.get("NB_OUT", "/content/drive/MyDrive/tqp-weights/probes/llama3.1-8b")
if not TEST:
    from google.colab import drive
    drive.mount("/content/drive")
for f in ("arms_results.jsonl", "codec_costs.jsonl"):
    assert os.path.exists(f"{REG}/{f}"), f"registered results missing in Drive: {REG}/{f}"
os.makedirs(f"{OUT}/out", exist_ok=True)
print("inputs", REG, "| outputs", OUT)

In [ ]:
# 3. The pinned packages (the NRP image's torch 2.8.0 with its torchvision 0.23.0, and the jobs' pins).
if not TEST:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "torch==2.8.0", "torchvision==0.23.0",
                    "transformers==4.56.1", "accelerate==1.14.0", "datasets==4.3.0", "safetensors", "numpy",
                    "huggingface_hub"], check=True)
PY = os.environ.get("NB_PY", sys.executable)
print(subprocess.run([PY, "-c", "import torch, transformers; print(torch.__version__, transformers.__version__)"],
                     capture_output=True, text=True, check=True).stdout)

In [ ]:
# 4. The code: a fresh full clone in a commit-named folder (never a leftover), and a step runner that streams output.
C = "b4646a9e11e3cb68b6d69666edaf39499b64979a"
CODE = f"{ROOT}/tqp_{C[:12]}"
if not os.path.exists(f"{CODE}/.git"):
    subprocess.run(["git", "clone", "-q", "https://github.com/ahb-sjsu/turboquant-pro", CODE], check=True)
subprocess.run(["git", "-C", CODE, "checkout", "-q", "--detach", C], check=True)
assert subprocess.run(["git", "-C", CODE, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip() == C
ENV = dict(os.environ, PYTHONPATH=f"{CODE}/benchmarks", PYTHONUNBUFFERED="1", HF_HUB_DISABLE_XET="1",
           PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", TOKENIZERS_PARALLELISM="false")


def step(args, label):
    print(">>", label, flush=True)
    p = subprocess.Popen([PY, "-m", *args], cwd=f"{CODE}/benchmarks", env=ENV,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        lines.append(line)
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"step failed ({p.returncode}): {label}" + chr(10) + "".join(lines[-30:]))


print("code", C[:12])

In [ ]:
# 5. The text, restaged and checked against the prereg.
import hashlib
TEXT = f"{ROOT}/wo/text"
step(["weight_observer.stage_text", "--dest", TEXT], "stage_text")
for n, h in {"train.txt": "aee724fa58bfbdeb3fc6803297fb6bab27b203d7c40b39ddef9b9770e5d52fe5",
             "test.txt": "696cca6b65a171b0a358a4be6732cdfdf2dd6164a32e20fd70e3c13fc4dfae83"}.items():
    assert hashlib.sha256(open(f"{TEXT}/{n}", "rb").read()).hexdigest() == h, n
print("text matches the prereg")

In [ ]:
# 6. The model at the prereg's mirror commit (16.1 GB), manifest checked.
import json
M = f"{ROOT}/wo/models/llama3.1-8b"
step(["weight_observer.stage_models", "--model-id", "unsloth/Meta-Llama-3.1-8B", "--revision", "e9a141a2091ea561b96483212645a2a05e6f99fc", "--dest", M],
     "stage_models")
assert json.load(open(f"{M}/STAGED.json"))["revision"] == "e9a141a2091ea561b96483212645a2a05e6f99fc"
print("model staged at the prereg's commit")

In [ ]:
# 7. The windows, tokenized once and checked against the registered hashes.
W = f"{ROOT}/wo/windows/llama3.1-8b"
if not os.path.exists(f"{W}/windows.pt"):
    step(["weight_observer.codec_run", "windows", "--model-path", M, "--text", TEXT, "--out", W], "windows")
h = json.load(open(f"{W}/hashes.json"))
assert h["calibration_windows"]["sha256"] == "7dcba79b76b0e185ab48d7e27697db01da6345122b6fd0f000a3d64ae9229db3", "calibration windows differ"
assert h["evaluation_windows"]["sha256"] == "2eda697bba5e21a5f25b95c50528b3325025ffafc2c7ef020a1e2211af163223", "scored windows differ"
print("windows match the registered hashes")

In [ ]:
# 8. The registered plans (git show, sha-checked), then the probes. VERIFY must print VERIFY_PASS.
PLANS = f"{ROOT}/llama3.1-8b.codec_arms.json"
blob = subprocess.run(["git", "-C", CODE, "show", f"{C}:benchmarks/weight_observer/planned/llama3.1-8b.codec_arms.json"],
                      capture_output=True, check=True).stdout
open(PLANS, "wb").write(blob)
assert hashlib.sha256(blob).hexdigest() == "df98535d971904026b4ad82cb4b50edf939d9bdf0ebdf0405f123d8cd0b81364", "the plans differ from the registered ones"
IN = ["--plans", PLANS, "--costs", f"{REG}/codec_costs.jsonl"]
GPU = ["--model-path", M, "--windows", f"{W}/windows.pt", "--cache", f"{ROOT}/probe_cache", "--lru-gib", "40"]
P = "weight_observer.probes"
step([P, "make", *IN, "--out", f"{OUT}/specs.json"], "make")
if not TEST:
    step([P, "cache", "--model-path", M, "--windows", f"{W}/windows.pt", "--specs", f"{OUT}/specs.json",
          "--cache", f"{ROOT}/probe_cache", "--out", "-"], "cache")
    step([P, "verify", *GPU, *IN, "--registered", f"{REG}/arms_results.jsonl", "--out", f"{OUT}/out"], "verify")
    step([P, "search", *GPU, *IN, "--out", f"{OUT}/out"], "search (64 steps)")
    step([P, "eval", *GPU, "--specs", f"{OUT}/specs.json", "--out", f"{OUT}/out"], "eval (flatness + single)")
    step([P, "score", *IN, "--registered", f"{REG}/arms_results.jsonl", "--specs", f"{OUT}/specs.json",
          "--probes", f"{OUT}/out", "--out", f"{OUT}/score.json"], "score")
print("STEP8_DONE")

In [ ]:
# 9. Check and hand-back.
if not TEST:
    v = json.load(open(f"{OUT}/out/verify.json")); s = json.load(open(f"{OUT}/score.json"))
    print("verify identical:", v["identical"], "| search steps:", s["search"]["steps"], "accepted:", s["search"]["accepted"])
    assert v["identical"] and s["search"]["steps"] == 64
    from google.colab import drive
    drive.flush_and_unmount()
print("DONE: MyDrive/tqp-weights/probes/llama3.1-8b (G: on the laptop)")